# Demo end-to-end — Telco Churn Prediction

Notebook consolidado mostrando o pipeline completo: do dataset cru à inferência via API. Reusa exatamente os mesmos módulos do pacote `churn` (sem código duplicado).

Sequência:

1. Descrição das features
2. Carga do dataset e verificação de qualidade
3. Resultados dos baselines vs MLP (já registrados no MLflow)
4. Operating points e threshold de produção
5. Inferência em um cliente exemplo via `ChurnPredictor`

## 1. Descrição das features

| Coluna | Tipo bruto | Domínio / valores | Descrição |
|---|---|---|---|
| `customerID` | `object` | `7590-VHVEG`, `5575-GNVDE`, … | Identificador único — descartado antes do treino. |
| `gender` | categórica | `Male`, `Female` | Gênero do cliente. Sinal fraco no modelo. |
| `SeniorCitizen` | inteiro | `0`, `1` | Flag binária para cliente sênior (definição etária exata não documentada na ficha do dataset; sources secundárias divergem entre 60 e 65 anos). Diferente de outros booleanos do dataset que usam `Yes/No`. |
| `Partner` | categórica | `Yes`, `No` | Tem cônjuge/parceiro. |
| `Dependents` | categórica | `Yes`, `No` | Tem dependentes (filhos, idosos sob cuidado, etc.). |
| `tenure` | inteiro | 0–72 (meses) | Tempo de relação com a operadora. **Fortemente correlacionado com churn** (curto = alto risco). |
| `PhoneService` | categórica | `Yes`, `No` | Possui serviço de telefonia. |
| `MultipleLines` | categórica | `Yes`, `No`, `No phone service` | Possui múltiplas linhas de telefone. |
| `InternetService` | categórica | `DSL`, `Fiber optic`, `No` | Tipo de internet contratado. **Fiber optic correlaciona com churn alto.** |
| `OnlineSecurity` | categórica | `Yes`, `No`, `No internet service` | Add-on de segurança online. Ausência aumenta churn. |
| `OnlineBackup` | categórica | `Yes`, `No`, `No internet service` | Add-on de backup online. |
| `DeviceProtection` | categórica | `Yes`, `No`, `No internet service` | Add-on de proteção de dispositivo. |
| `TechSupport` | categórica | `Yes`, `No`, `No internet service` | Add-on de suporte técnico. Ausência aumenta churn. |
| `StreamingTV` | categórica | `Yes`, `No`, `No internet service` | Streaming de TV. |
| `StreamingMovies` | categórica | `Yes`, `No`, `No internet service` | Streaming de filmes. |
| `Contract` | categórica | `Month-to-month`, `One year`, `Two year` | Tipo de contrato. **Feature mais informativa.** Month-to-month tem ~43% churn. |
| `PaperlessBilling` | categórica | `Yes`, `No` | Fatura sem papel. |
| `PaymentMethod` | categórica | `Electronic check`, `Mailed check`, `Bank transfer (automatic)`, `Credit card (automatic)` | Método de pagamento. Electronic check correlaciona com churn alto. |
| `MonthlyCharges` | float | 18.25–118.75 (USD) | Cobrança mensal. |
| `TotalCharges` | string→float | 0–8684.80 (USD) | Cobrança total acumulada. **Chega como string** com whitespace para 11 contratos novos — coagido para float no preprocessing. |
| `Churn` | categórica | `Yes`, `No` | **Target.** Codificado para 1/0 antes do treino. |

**Feature engineering adicionada**:

- `tenure_bin`: discretização de `tenure` em 4 faixas (`(-0.07, 18]`, `(18, 36]`, `(36, 54]`, `(54, 72]`). Ajuda modelos lineares a capturar a não-linearidade do tempo de contrato.

**Output do preprocessing**: 19 features brutas (sem `customerID` e `Churn`) → 49 colunas numéricas após one-hot encoding e adição de `tenure_bin`.

## 2. Carga do dataset

In [1]:
import json
from pathlib import Path

import pandas as pd
import torch
from sklearn.model_selection import train_test_split

from churn.config import MODELS_DIR, RANDOM_SEED
from churn.dataset.loader import load_raw_dataset
from churn.dataset.preprocessing import build_preprocessor, split_features_target
from churn.modeling.evaluation import compute_classification_metrics, operating_point_table
from churn.modeling.predictor import ChurnPredictor
from churn.modeling.training import predict_proba

In [2]:
df = load_raw_dataset()
print(f"Linhas: {df.shape[0]:,} | Colunas: {df.shape[1]}")
churn_rate = (df["Churn"] == "Yes").mean()
print(f"Taxa de churn: {churn_rate:.2%}")

Linhas: 7,043 | Colunas: 21
Taxa de churn: 26.54%


## 3. Resultados — comparação MLP vs baselines

Os baselines (Dummy, Logistic Regression, Random Forest) e o MLP foram avaliados com **5-fold stratified CV** sobre o mesmo split de treino. Reproduzimos a tabela final dos artefatos persistidos.

In [3]:
metadata = json.loads((MODELS_DIR / "metadata.json").read_text())
print("Modelo final em produção:")
print(f"  arquitetura:  {metadata['hidden_dims']}")
print(f"  dropout:      {metadata['dropout']}")
print(f"  lr:           {metadata['lr']}")
print(f"  threshold:    {metadata['production_threshold']:.4f}")

Modelo final em produção:
  arquitetura:  [128, 64]
  dropout:      0.3
  lr:           0.0005
  threshold:    0.4399


In [4]:
# Tabela comparativa (CV-averaged - valores publicados pelos runs do MLflow)
comparison = pd.DataFrame(
    {
        "ROC AUC": [0.500, 0.846, 0.839, 0.843],
        "PR AUC":  [0.265, 0.660, 0.644, 0.632],
        "F1":      [0.000, 0.625, 0.612, 0.628],
        "Recall":  [0.000, 0.801, 0.646, 0.848],
        "Accuracy":[0.735, 0.745, 0.783, 0.730],
    },
    index=["dummy_majority", "logistic_regression", "random_forest", "mlp_tuned"],
)
comparison.style.format("{:.3f}").background_gradient(cmap="RdYlGn", axis=0)

,ROC AUC,PR AUC,F1,Recall,Accuracy
dummy_majority,0.500,0.265,0.000,0.000,0.735
logistic_regression,0.846,0.660,0.625,0.801,0.745
random_forest,0.839,0.644,0.612,0.646,0.783
mlp_tuned,0.843,0.632,0.628,0.848,0.730


**Leitura honesta**: regressão logística e MLP empatam em ROC AUC (~0,84). O sinal nesse dataset é majoritariamente linear — a capacidade extra do MLP só recupera ganho marginal. O MLP entra como modelo central conforme escopo, mas a logistic regression seria igualmente competitiva em produção.

## 4. Operating points (test set holdout)

In [5]:
op = metadata["operating_points"]
op_df = pd.DataFrame(op).T[["threshold", "precision", "recall", "f1", "accuracy", "roc_auc"]]
op_df.style.format("{:.4f}").background_gradient(cmap="RdYlGn", subset=["recall", "f1", "roc_auc"], axis=0)

,threshold,precision,recall,f1,accuracy,roc_auc
default_0.5,0.5000,0.5086,0.7914,0.6192,0.7417,0.8431
max_f1,0.4399,0.4945,0.8476,0.6246,0.7296,0.8431
recall_min_0.85,0.4355,0.4915,0.8503,0.6229,0.7268,0.8431
cost_optimal_1_to_5,0.3700,0.4680,0.8984,0.6154,0.7019,0.8431


**Threshold de produção** = `max_f1` (0,44) — escolhido por maximizar F1 (média harmônica de precision e recall) no test set.

A tabela explicita o trade-off precision × recall — qualquer time consumindo a API pode aplicar o seu próprio cutoff sobre `churn_probability` e operar em outro ponto sem retreinar.

## 5. Inferência em um cliente exemplo

In [6]:
predictor = ChurnPredictor.from_artifacts()

cliente_alto_risco = {
    "gender": "Female",
    "SeniorCitizen": 0,
    "Partner": "No",
    "Dependents": "No",
    "tenure": 2,
    "PhoneService": "Yes",
    "MultipleLines": "No",
    "InternetService": "Fiber optic",
    "OnlineSecurity": "No",
    "OnlineBackup": "No",
    "DeviceProtection": "No",
    "TechSupport": "No",
    "StreamingTV": "Yes",
    "StreamingMovies": "Yes",
    "Contract": "Month-to-month",
    "PaperlessBilling": "Yes",
    "PaymentMethod": "Electronic check",
    "MonthlyCharges": 95.50,
    "TotalCharges": 191.00,
}

cliente_baixo_risco = {
    "gender": "Male",
    "SeniorCitizen": 0,
    "Partner": "Yes",
    "Dependents": "Yes",
    "tenure": 60,
    "PhoneService": "Yes",
    "MultipleLines": "Yes",
    "InternetService": "DSL",
    "OnlineSecurity": "Yes",
    "OnlineBackup": "Yes",
    "DeviceProtection": "Yes",
    "TechSupport": "Yes",
    "StreamingTV": "No",
    "StreamingMovies": "No",
    "Contract": "Two year",
    "PaperlessBilling": "No",
    "PaymentMethod": "Bank transfer (automatic)",
    "MonthlyCharges": 65.00,
    "TotalCharges": 3900.00,
}

df_clientes = pd.DataFrame([cliente_alto_risco, cliente_baixo_risco])
probas, decisions = predictor.predict_dataframe(df_clientes)

result = pd.DataFrame(
    {
        "perfil": ["alto_risco_curto_prazo", "baixo_risco_longo_prazo"],
        "probability": [round(p, 4) for p in probas],
        "decision": decisions,
    }
)
result

,perfil,probability,decision
0,alto_risco_curto_prazo,0.8885,True
1,baixo_risco_longo_prazo,0.0513,False


Como esperado:

- **Cliente alto risco** (2 meses, fibra óptica, contrato mensal, electronic check) — probabilidade alta, classificado como churn.
- **Cliente baixo risco** (60 meses, contrato 2 anos, débito automático, todos add-ons) — probabilidade baixa, classificado como não-churn.

A mesma chamada em produção é exposta via `POST /predict` na API (ver `docs/deploy.md`).

## Resumo

- **Dataset**: 7.043 clientes, 21 colunas, 26,5% churn — desbalanceado mas longe de raro.
- **Pipeline**: split estratificado 80/20 → preprocessing sklearn (impute + scale + one-hot + `tenure_bin`) → MLP PyTorch (128, 64) com early stopping.
- **MLP final**: ROC AUC 0,843, F1 0,628 no operating point de produção (threshold 0,44).
- **Comparado a baselines**: empata com regressão logística; supera Random Forest em recall.
- **Threshold tuning**: 4 operating points expostos para diferentes prioridades de negócio.
- **Limitação documentada**: probabilidades overconfident — usar como score de prioridade, não como número absoluto.

Próximos passos viáveis: features comportamentais, calibração via temperature scaling, retraining mensal disparado por drift.